# SmartNest Security Assessment Plan

This notebook records the assessment plan for the SmartNest coursework project.

Project focus:
- containerised smart home simulation
- threat modelling and attack demonstration
- defensive redesign and validation

Environment assumption:
All components run in Docker containers on a local development machine rather than on Raspberry Pi or ESP32 hardware.


---

## 整体场景设定

你们模拟一个 **"SmartNest" 智能家居平台**，包含以下组件：

**中心 Hub**：Docker 容器中的 MQTT Broker + Web 管理面板
**设备节点**：Python 模拟设备容器，分别模拟智能门锁、智能灯和智能闹钟
**通信协议**：MQTT（设备↔Broker）、HTTP/HTTPS（用户↔Web 面板）
**部署方式**：通过 Docker Compose 在本地电脑上完成一键编排和复现实验环境

---

## Coursework 1：威胁建模 & 攻击模拟

### 威胁 1：MQTT 中间人攻击（MitM on MQTT）

**原理**：很多智能家居系统仍使用未加密的 MQTT（端口 1883），攻击者一旦能够访问相同网络或宿主机开放端口，就可以嗅探和伪造消息。

**具体实现步骤**：
1. 在 Docker 中部署 Mosquitto Broker，**故意不开 TLS**，并开放 1883 端口
2. 设备模拟器容器持续发布状态到 `home/lock/status`、`home/light/status`、`home/alarm/status`，并订阅对应 command topic
3. 攻击者在宿主机或另一容器中使用 `mosquitto_sub` 和 `mosquitto_pub` 监听与伪造 MQTT 消息
4. 用 Wireshark 或 `tcpdump` 抓包，展示明文 MQTT payload（例如开锁指令、设备状态、用户名字段）
5. 用 `mosquitto_pub` 伪造一条开锁指令：`mosquitto_pub -h localhost -t home/lock/command -m '{"action":"UNLOCK"}'`，观察设备状态变化

**交付物**：Wireshark 截图、攻击脚本、攻击链流程图

### 威胁 2：Web 管理面板漏洞（认证绕过 + 注入）

**原理**：智能家居 Hub 通常有一个 Web 管理界面，如果开发不当，容易存在弱认证、SQL 注入、XSS 等问题。

**具体实现步骤**：
1. 在 Docker 容器中运行 Flask 管理面板（设备列表、用户登录、设备控制页面）
2. **故意留漏洞**：默认密码 admin/admin、SQL 注入点（登录表单）、未做 CSRF 保护的设备控制接口
3. 演示攻击：用 `sqlmap` 跑注入拿到用户表、用默认密码登录、通过 CSRF 构造恶意页面远程控制设备
4. 可选加分项：演示 XSS 存储型攻击——在设备名称字段注入脚本，管理员查看时触发

**交付物**：漏洞利用脚本、sqlmap 输出截图、CSRF PoC HTML 页面

### 威胁 3（可选加分）：容器逃逸 / 镜像配置暴露

如果团队有余力，可以进一步分析 Docker 镜像、Compose 配置和容器间网络暴露面，例如弱默认配置、明文环境变量、过宽的端口暴露或缺少认证的内部 API。

### 风险矩阵

用一个 5×5 的风险矩阵（Likelihood × Impact）对威胁排优先级，MQTT MitM 是"高可能 × 高影响"，Web 漏洞是"中可能 × 高影响"。

---

## Coursework 2：防御方案

### 防御 1：MQTT over TLS + 设备证书认证

1. 给 Mosquitto 配置 TLS（生成 CA 证书、服务端证书、客户端证书）
2. 为每个设备模拟客户端分配唯一客户端证书，Broker 开启 `require_certificate true`
3. 演示：攻击者再次尝试嗅探 → Wireshark 只看到加密流量；伪造消息 → 被 Broker 拒绝连接
4. 代码展示证书生成脚本和 Python MQTT 客户端的 TLS 连接配置

### 防御 2：AI 驱动的入侵检测系统（IDS）

1. 在 Hub 上部署一个轻量 Python IDS 脚本
2. 用正常运行时的 MQTT 流量训练一个基线模型（可以用 Isolation Forest 或简单的统计阈值）
3. 检测异常：突然大量 publish 请求、非正常时间段的开锁指令、陌生 Client ID 连接
4. 触发告警（邮件/Telegram 通知 + Web 面板红色警告）
5. 展示对比：无 IDS 时攻击成功 vs 有 IDS 时 30 秒内检测到并阻断

### 防御 3：Web 面板加固

1. 用参数化查询消除 SQL 注入
2. 实现 CSRF Token
3. 加入 MFA（TOTP，用 `pyotp` 库实现）
4. 演示：加固后再跑 sqlmap → 无果；没有 TOTP 码无法登录

### 防御 4：安全 OTA 固件更新

1. 将其调整为**安全镜像/配置更新机制**：对容器镜像或更新包进行签名校验（用 RSA/Ed25519）
2. 演示：篡改过的更新包或配置文件被系统拒绝加载

---

## 4 人分工建议

| 成员 | 负责模块 | 具体任务 |
|------|---------|---------|
| A | MQTT 攻防 | 搭建 MQTT 环境、MitM 攻击演示、TLS 防御实现 |
| B | Web 安全 | 搭建 Flask 面板、留漏洞+攻击演示、加固+MFA |
| C | IDS 系统 | 流量采集、异常检测模型训练、告警系统 |
| D | 架构 + 集成 | Docker 架构设计、Compose 编排、安全更新机制、报告整合 |

每个人的工作互相依赖（比如 C 的 IDS 要接 A 的 MQTT 流量），但又可以独立开发测试，最后集成。

---

## 实验环境与资源需求

本项目不依赖树莓派、ESP32 或其他实体硬件。实验环境基于本地电脑上的 Docker Desktop / Docker Engine 搭建，包含 Mosquitto 容器、Web 面板容器和 Python 设备模拟器容器。攻击端可直接使用宿主机工具，或者额外启一个攻击容器来复现监听、伪造消息和漏洞利用流程。

---



为了让项目更容易复现、演示和提交，建议你们统一采用 **Docker 容器模拟环境**，不再依赖树莓派或 ESP32 实体硬件。下面是一套更适合当前 repo 的容器化实施方案。

---

## 你们需要准备什么

**必需软件环境：**

一台笔记本电脑即可，安装好 Docker Desktop（或 Docker Engine + Docker Compose）、浏览器，以及基础安全测试工具。对于攻击演示，宿主机可以安装 `mosquitto-clients`、`sqlmap`、Wireshark，也可以额外启动一个攻击容器来执行订阅、发布和渗透测试。

**可选环境：** 如果你们希望把攻击环境隔离出来，可以使用 Kali Linux 虚拟机，或者在 Docker 网络中再加入一个 attacker 容器。

---

## 容器架构

整体结构是这样的：Docker Compose 创建一个内部网络，其中 Mosquitto 作为 Broker，Flask 作为 Web 面板，Python 脚本作为设备模拟器。攻击者既可以从宿主机通过映射端口接入，也可以加入同一个 Docker 网络。这样能较稳定地重现“弱认证 Web 面板 + 明文 MQTT + 可伪造设备指令”的攻击链。

---

## 具体搭建步骤

### 第一步：配置容器化 Hub（第 1 天上午）

在本地电脑上准备 `docker-compose.yml`，定义 Mosquitto Broker、Flask Web 面板和设备模拟器三个服务。Broker 初始配置保持不安全：不开 TLS、不设 MQTT 用户认证、开放 1883 端口，方便 Coursework 1 做监听和伪造攻击。

记录对外暴露的端口，例如 Web 面板 `localhost:5001`、MQTT Broker `localhost:1883`，后续攻击和演示都围绕这些接口展开。

### 第二步：实现设备模拟器（第 1 天下午）

用 Python 编写一个设备模拟器进程或容器，模拟多个智能设备的状态机和命令处理逻辑。

**门锁模拟器：** 订阅 MQTT topic `home/lock/command`，收到 `"UNLOCK"` 就把状态切换为 `UNLOCKED`，并发布到 `home/lock/status`。

**其他设备模拟器：** 智能灯和智能闹钟通过相同方式运行，周期性上报状态，并根据控制指令更新内部状态。

核心逻辑就是 MQTT 连接 → 订阅 command topic → 循环发布 status / event topic。这样既保留 IoT 行为模型，也更容易调试和复现实验。

### 第三步：搭建 Web 管理面板（第 2 天）

在单独的 Flask 容器中实现一个简单的管理界面。首页显示所有设备的实时状态，有按钮可以发送控制指令，并提供登录页面。这个面板就是 Coursework 1 里被攻击的目标之一，也是 Coursework 2 里被加固的对象。

### 第四步：联调验收（第 2-3 天）

启动全部容器后，打开三个终端窗口分别观察：一个看 Mosquitto 日志，一个用 `mosquitto_sub -t 'home/#'` 订阅所有 MQTT 消息，一个打开浏览器访问 Flask 面板。在面板上点“开锁”后，验证设备模拟器状态是否变化，并确认新的状态消息已经被 Web 面板接收。这条链路打通后，环境就搭好了。

---

## 为什么选择 Docker

对 coursework 来说，Docker 最大的优势是**可复现**、**易分享** 和 **易重置**。所有成员都可以在自己的电脑上拉起同样的环境，不会被硬件借用、网络配置或烧录问题卡住。

另外，Docker 也更适合做攻击与防御前后的对比：你们可以快速替换 broker 配置、切换 insecure / hardened 版本、重放攻击脚本，并保留日志和抓包证据，便于写报告和录视频。

---

## 现在具体要做的事

你们现在最应该做的是先把容器化基础环境跑通：确认 Docker 能正常启动 broker、web 和 devices 三个服务；确认浏览器可以访问面板；确认 `mosquitto_sub` 能看到明文 MQTT 消息；然后再基于这套环境补齐攻击脚本、TLS、防护和 IDS。

如果需要，我下一步可以继续帮你把这份 notebook 再整理成更正式的 coursework 风格，比如加上目标、实验范围、交付物、时间线和风险说明。

先给你一个整体的协议架构图，然后列出每个设备的详细 JSON 规范。每个设备遵循统一的三类 topic：`status`（设备上报状态）、`command`（面板下发指令）、`event`（设备上报事件告警）。下面是完整 JSON 规范。

---

## 公共字段（所有消息都带）

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200000,
  "version": "1.0"
}
```

`device_type` 固定三个值：`"lock"` / `"light"` / `"alarm"`。`version` 是协议版本号，方便以后扩展。

---

## 1. 智能门锁 (Smart Lock)

**`home/lock/status`** — 门锁每 10 秒上报一次

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200000,
  "version": "1.0",
  "state": "LOCKED",
  "battery": 85,
  "method": "auto",
  "last_user": "admin"
}
```

`state` 取值：`"LOCKED"` / `"UNLOCKED"`
`method` 记录最近一次操作方式：`"manual"` / `"remote"` / `"auto"` / `"pin"`
`last_user` 记录最近操作人

**`home/lock/command`** — Web 面板下发

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200020,
  "version": "1.0",
  "action": "UNLOCK",
  "issued_by": "admin",
  "pin": "1234"
}
```

`action` 取值：`"LOCK"` / `"UNLOCK"` / `"SET_PIN"`
`pin` 仅在 `SET_PIN` 时必填，其他时候可选（用于验证身份）

**`home/lock/event`** — 异常事件触发时发送

```json
{
  "device_id": "lock_01",
  "device_type": "lock",
  "timestamp": 1712200030,
  "version": "1.0",
  "event": "TAMPER_DETECTED",
  "severity": "high",
  "detail": "Vibration sensor triggered"
}
```

`event` 取值：`"TAMPER_DETECTED"` / `"LOW_BATTERY"` / `"FORCED_ENTRY"` / `"PIN_FAILED"`
`severity`：`"low"` / `"medium"` / `"high"` / `"critical"`

---

## 2. 智能灯 (Smart Light)

**`home/light/status`** — 每 10 秒上报

```json
{
  "device_id": "light_01",
  "device_type": "light",
  "timestamp": 1712200000,
  "version": "1.0",
  "power": "ON",
  "brightness": 80,
  "color_temp": 4000,
  "rgb": [255, 180, 100],
  "mode": "manual"
}
```

`power`：`"ON"` / `"OFF"`
`brightness`：0-100 整数
`color_temp`：色温（开尔文），2700 暖光 ~ 6500 冷光
`rgb`：RGB 值，和 `color_temp` 二选一使用
`mode`：`"manual"` / `"schedule"` / `"auto"`（自动亮度）

**`home/light/command`** — 控制指令

```json
{
  "device_id": "light_01",
  "device_type": "light",
  "timestamp": 1712200020,
  "version": "1.0",
  "action": "SET_BRIGHTNESS",
  "issued_by": "admin",
  "params": {
    "brightness": 50
  }
}
```

`action` 取值：`"ON"` / `"OFF"` / `"SET_BRIGHTNESS"` / `"SET_COLOR"` / `"SET_MODE"`
`params` 根据 action 不同而变化：
- `SET_BRIGHTNESS` → `{"brightness": 50}`
- `SET_COLOR` → `{"rgb": [255, 0, 0]}` 或 `{"color_temp": 3000}`
- `SET_MODE` → `{"mode": "schedule", "schedule": {"on": "18:00", "off": "23:00"}}`

**`home/light/event`** — 异常事件

```json
{
  "device_id": "light_01",
  "device_type": "light",
  "timestamp": 1712200030,
  "version": "1.0",
  "event": "OVERHEATING",
  "severity": "high",
  "detail": "Temperature reached 85C, auto shutdown"
}
```

`event` 取值：`"OVERHEATING"` / `"BULB_FAILURE"` / `"POWER_SURGE"`

---

## 3. 智能闹钟 (Smart Alarm Clock)

**`home/alarm/status`** — 每 10 秒上报

```json
{
  "device_id": "alarm_01",
  "device_type": "alarm",
  "timestamp": 1712200000,
  "version": "1.0",
  "alarm_time": "07:30",
  "enabled": true,
  "ringing": false,
  "volume": 70,
  "repeat": ["MON", "TUE", "WED", "THU", "FRI"],
  "snooze_count": 0
}
```

`alarm_time`：24 小时制字符串
`repeat`：重复日期数组，空数组 `[]` 表示仅一次
`snooze_count`：当前已贪睡的次数

**`home/alarm/command`** — 控制指令

```json
{
  "device_id": "alarm_01",
  "device_type": "alarm",
  "timestamp": 1712200020,
  "version": "1.0",
  "action": "SET_ALARM",
  "issued_by": "admin",
  "params": {
    "alarm_time": "06:45",
    "repeat": ["MON", "WED", "FRI"],
    "volume": 80
  }
}
```

`action` 取值：`"SET_ALARM"` / `"ENABLE"` / `"DISABLE"` / `"DISMISS"` / `"SNOOZE"`
- `DISMISS` 和 `SNOOZE` 不需要 `params`
- `SNOOZE` 默认延迟 5 分钟（设备端硬编码）

**`home/alarm/event`** — 事件通知

```json
{
  "device_id": "alarm_01",
  "device_type": "alarm",
  "timestamp": 1712200030,
  "version": "1.0",
  "event": "ALARM_TRIGGERED",
  "severity": "low",
  "detail": "Alarm ringing at 07:30"
}
```

`event` 取值：`"ALARM_TRIGGERED"` / `"ALARM_MISSED"` / `"MAX_SNOOZE_REACHED"`

---

## 为什么这样设计

统一的三 topic 模式（status / command / event）让每个设备的行为可预测，IDS 做异常检测时规则统一。`params` 字段把不同指令的参数嵌套在一起，避免顶层字段爆炸。`severity` 字段让告警系统可以按严重程度分级处理。

攻击演示的时候，这套 JSON 格式在 Wireshark 里抓包特别直观——攻击者一眼就能看到 `"action": "UNLOCK"` 和 `"issued_by": "admin"`，完美展示明文传输的危险性。

要我接下来把这个协议写成一份正式的 Markdown 规范文档放到你们的 GitHub repo 里？